#Explore QA RAG Chains with LCEL

In [1]:
from langchain_openai import OpenAIEmbeddings

# details here: https://openai.com/blog/new-embedding-models-and-api-updates
openai_embed_model = OpenAIEmbeddings(model='text-embedding-3-small')

In [2]:
docs = [
 'Quantum mechanics describes the behavior of very small particles.',
 'Photosynthesis is the process by which green plants make food using sunlight.',
 'Artificial Intelligence aims to create machines that can think and learn.',
 'The pyramids of Egypt are historical monuments that have stood for thousands of years.',
 'New Delhi is the capital of India and the seat of all three branches of the Government of India.',
 'Biology is the study of living organisms and their interactions with the environment.',
 'Music therapy can aid in the mental well-being of individuals.',
 'Mumbai is the financial capital and the most populous city of India. It is the financial, commercial, and entertainment capital of South Asia.',
 'The Milky Way is just one of billions of galaxies in the universe.',
 'Economic theories help understand the distribution of resources in society.',
 'Kolkata is the de facto cultural capital of India and a historically and culturally significant city. Calcutta served as the de facto capital of India until 1911.',
 'Yoga is an ancient practice that involves physical postures and meditation.'
]

In [3]:
from langchain_chroma import Chroma

# create vector DB of docs and embeddings - takes 1 min on Colab
chroma_db = Chroma.from_texts(texts=docs, collection_name='db_docs',
                              # need to set the distance function to cosine else it uses euclidean by default
                              # check https://docs.trychroma.com/guides#changing-the-distance-function
                              collection_metadata={"hnsw:space": "cosine"},
                              embedding=openai_embed_model)

In [4]:
similarity_threshold_retriever = chroma_db.as_retriever(search_type="similarity_score_threshold",
                                                        search_kwargs={"k": 3,
                                                                       "score_threshold": 0.3})

In [5]:
query = "what is the capital of India?"
top3_docs = similarity_threshold_retriever.invoke(query)
top3_docs

[Document(id='d140039d-9063-4b5b-8fed-44a43a368609', metadata={}, page_content='New Delhi is the capital of India and the seat of all three branches of the Government of India.'),
 Document(id='08d75b99-e189-4325-b155-eb65af5c822a', metadata={}, page_content='Mumbai is the financial capital and the most populous city of India. It is the financial, commercial, and entertainment capital of South Asia.'),
 Document(id='cc018ee5-65f5-43a7-a8be-9d0e529aef3a', metadata={}, page_content='Kolkata is the de facto cultural capital of India and a historically and culturally significant city. Calcutta served as the de facto capital of India until 1911.')]

#Build a QA RAG Chain
##To build a RAG chain we need a prompt template which instructs the LLM to not answer questions beyond the scope of the retrieved context documents, there are various such prompts out there, we craft one ourselves below

In [6]:
from langchain_core.prompts import ChatPromptTemplate

prompt = """You are an assistant for question-answering tasks.
            Use the following pieces of retrieved context to answer the question.
            If no context is present or if you don't know the answer, just say that you don't know.
            Do not make up the answer unless it is there in the provided context.
            Keep the answer concise and to the point with regard to the question.

            Question:
            {question}

            Context:
            {context}

            Answer:
         """

prompt_template = ChatPromptTemplate.from_template(prompt)

In [7]:
from langchain_openai import ChatOpenAI

chatgpt = ChatOpenAI(model_name='gpt-4o-mini', temperature=0)

LCEL Syntax for QA RAG Chain - Recommended

Here we show you how to create the RAG chain using LangChain's recommended LCEL.

Note "context": (similarity_threshold_retriever | RunnableLambda(format_docs)) we have wrap in the RunnableLambda, it will also work if do not wrap this isn to RunnableLambda because LangChain automatically wraps simple callables into RunnableLambda in linear pipelines. but is is good practice to use RunnableLambda.

In [10]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableLambda

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_rag_chain = (
    {
        "context": (similarity_threshold_retriever | RunnableLambda(format_docs)), # first retriever will retrive the node then it goes to format_docs function that 
                                                            # will get the page content from each retrived document(page1  content /n/n page2 content)
        "question": RunnablePassthrough()
    }
      |
    prompt_template
      |
    chatgpt
)

In [11]:
query = "What is the capital of India?"
result = qa_rag_chain.invoke(query)
print(result.content)

New Delhi is the capital of India.


In [12]:
query = "Tell me about the financial capital of India in detail"
result = qa_rag_chain.invoke(query)
print(result.content)

Mumbai is the financial capital of India, as well as the most populous city in the country. It serves as the financial, commercial, and entertainment capital of South Asia.


In [13]:
query = "What is the fastest animal?"
result = qa_rag_chain.invoke(query)
print(result.content)

No relevant docs were retrieved using the relevance score threshold 0.3


I don't know.
